# Encoder router — shakedown (dataset v2)

Runs the d63 prototype end to end: build the corpus-branch artifacts, assemble the
training table, smoke one fold, then sweep every arm through leave-one-lane-out CV.

**Every number in this notebook is shakedown tier.** The catalog is stale in columns
several cell predicates read (d56e/d62 re-extraction owed), so cell targets are wrong
for digit-bearing queries. The arm comparison of record waits for the v3 dataset build.

Prerequisites: `poetry install` (torch + lightgbm are new) and a populated
`labels.parquet` — it carries query text and cell itself.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from encoder_router.table import NgramSvd
from encoder_router.targets import CorpusProfile, GoldDocProfile
from encoder_router.training import QueryEmbeddings, TrainingTable
from encoder_router.evaluate import BGE, E5, Arm, LaneCV, run_arms
from encoder_router.targets import OUT_DIR

pd.set_option("display.width", 160)
table = TrainingTable()

In [ ]:
table.route_targets()

## 1 · The training frame

Acceptability labels joined with query text and cell. Expect a printed count of
labelled rows dropped for missing text — rows the selection no longer carries.

In [ ]:
frame = table.frame
print(f"{len(frame):,} rows across {frame['dataset'].nunique()} lanes")
display(frame["shape"].value_counts())
ok_columns = [c for c in frame.columns if c.startswith("ok_")]
display(frame[ok_columns].mean().rename("acceptability rate"))
frame.groupby("dataset").size().sort_values(ascending=False)

### 1b · Label hygiene: the min_relevance audit

Graded-qrel lanes where `min_relevance = 1` count weakly-related documents as
full successes (the nfcorpus "aneurysm → L-citrulline" tie) — every route finds
*some* weak positive and the row ties. High `share_grade_1` + high `tie_share`
marks a lane whose ties are manufactured by the relevance floor, not by
retrieval. Raising a lane's `min_relevance` in `lanes.py` changes its labels,
so that lane must be relabelled afterwards:
`poetry run python src/scripts/label_routes.py --only <lane> --force`.

In [ ]:
from pathlib import Path

from hybrid_search_rrf_dataset.lanes import LANES

records = []
for lane in sorted(frame["dataset"].unique()):
    source = LANES[lane].source.name if lane in LANES else lane
    qrels = pd.read_parquet(Path("data") / source / "qrels.parquet")
    grades = qrels["relevance"].value_counts().sort_index()
    if grades.index.max() <= 1:
        continue
    lane_rows = frame[frame["dataset"] == lane]
    records.append({
        "lane": lane,
        "min_relevance": LANES[lane].min_relevance if lane in LANES else 1,
        "grades": grades.to_dict(),
        "share_grade_1": float((qrels["relevance"] == 1).mean()),
        "tie_share": float((lane_rows["shape"] == "all_tied").mean()),
        "rows": len(lane_rows),
    })
audit = pd.DataFrame(records).sort_values("share_grade_1", ascending=False)
audit.round(3)

## 2 · Corpus-branch artifacts

Both builders are incremental per lane — a crashed run resumes where it stopped.
`GoldDocProfile` is the slow one: the full extractor over every unique gold document.

In [ ]:
lanes = tuple(sorted(frame["dataset"].unique()))
corpus_profile = CorpusProfile(lanes).build()
corpus_profile.round(3)

In [ ]:
gold_profile = GoldDocProfile(frame).build()
print(f"{len(gold_profile):,} query rows, "
      f"{gold_profile['gold.overlap'].isna().mean():.1%} without overlap")
gold_profile.head()

## 3 · Target sanity

Cell targets are every predicate evaluated on every row — check which archetypes have
support, and that the two assumed column names exist before anything trains on them.

In [ ]:
assert "natural_language_signal.natural_language_share" in table.catalog_rows.columns, (
    "NL scalar column name differs — fix encoder_router.targets.NL_SHARE"
)

cells = table.cell_targets
support = cells.sum().sort_values()
print(f"{int((support == 0).sum())} of {len(support)} cells with zero positive rows")
print(f"rows matching no cell: {(cells.sum(axis=1) == 0).mean():.1%}")
display(support.head(10).rename("thinnest"))
display(support.tail(10).rename("fattest"))

corpus_targets = table.corpus_targets()
corpus_targets.isna().mean().sort_values(ascending=False).head(8).rename("NaN share")

## 4 · Embedding caches

One forward pass per model, cached to parquet. The e5 cell feeds arm `e5_control` —
skip it if you are not running that arm yet.

In [ ]:
embeddings = QueryEmbeddings(BGE).matrix(frame)
embeddings.shape

In [ ]:
QueryEmbeddings(E5).matrix(frame).shape

## 5 · One-fold smoke

The design arm against a single held lane — cheap proof that the whole path runs
before committing to the full sweep.

In [ ]:
cv = LaneCV(table)
cv.run(Arm("design"), lanes=("beir-nfcorpus",))

## 6 · Full arm sweep

Every arm through every leave-one-lane-out fold. Results persist so readouts can be
re-run without re-training.

In [ ]:
# A diverse 10-lane holdout panel: big/small, web/code/math/bio/legal-ish,
# identifier-dense and identifier-free. Training rows stay FULL per fold —
# only the number of readout points shrinks.
PANEL = (
    "beir-nfcorpus", "msmarco-passage-dev", "rarb-math",
    "crumb-code-retrieval", "gooaq", "scirgen-geo-en",
    "lotte-technology-forum", "quest", "freshstack-godot", "limit",
)
# _v3: captured-score threshold tuning (per-head), route pos_weight, and
# all_tied rows down-weighted to 0.25 in the route loss. _v2 = agreement-tuned
# shared threshold; unsuffixed = the collapsed 0.5 run. Both kept for reference.
RESULTS_PATH = OUT_DIR / "arm_results_panel__hedge_v3.parquet"

results = run_arms(table, lanes=PANEL, out_path=RESULTS_PATH)
results.head()

## 7 · Readout

`differ_agreement` is the headline: agreement with the serve oracle exactly where the
routes disagree. The spread across lanes matters as much as the mean — ~15 lanes is
the real sample size. Read the design arm against `no_branches` (do the branches
help?), `shuffled_targets` (information or regularization?), and `lightgbm` (is the
MLP needed at all?).

In [ ]:
results = pd.read_parquet(RESULTS_PATH)
wanted = [
    "differ_agreement", "serve_agreement", "captured", "serve_captured",
    "const_dense", "oracle",
] + [c for c in results.columns if c.startswith(("threshold_", "served_"))]
present = [c for c in wanted if c in results.columns]
summary = results.groupby("arm")[present].mean()
summary.insert(1, "differ_spread", results.groupby("arm")["differ_agreement"].std())
summary.sort_values("captured", ascending=False).round(4)

import matplotlib.pyplot as plt

arms = sorted(results["arm"].unique())
fig, ax = plt.subplots(figsize=(9, 4))
for i, arm in enumerate(arms):
    rows = results[results["arm"] == arm]
    ax.scatter([i] * len(rows), rows["differ_agreement"], alpha=0.5)
    ax.scatter([i], [rows["differ_agreement"].mean()], color="black", marker="_", s=400)
ax.set_xticks(range(len(arms)), arms, rotation=20)
ax.set_ylabel("differ_agreement per held lane")
ax.set_title("leave-one-lane-out spread (mean marked)")
plt.tight_layout()
plt.show()

In [ ]:
# paired per-lane deltas on CAPTURED score — the metric the tuner now
# optimizes; agreement rewards the majority-class constant by construction.
wide = results.pivot(index="lane", columns="arm", values="captured")
wide["constant_sparse"] = (
    frame[frame["serve"].notna()]
    .groupby("dataset")["score_sparse_only"].mean()
)
pairs = {
    "no_branches - design": wide["no_branches"] - wide["design"],
    "design - shuffled": wide["design"] - wide["shuffled_targets"],
    "e5 - design": wide["e5_control"] - wide["design"],
    "best_arm - constant": (
        wide.drop(columns="constant_sparse").max(axis=1)
        - wide["constant_sparse"]
    ),
}
pd.DataFrame(
    {k: {"mean": v.mean(), "lanes_won": (v > 0).sum()} for k, v in pairs.items()}
).T

## 8 · Archetype probes

The seven standing probes through a design-arm router trained on all rows (no holdout —
this is the smoke test, not an evaluation).

In [ ]:
from sentence_transformers import SentenceTransformer

from encoder_router.evaluate import tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import EMBEDDING_PREFIXES
from hybrid_search_rrf_dataset.probes import ARCHETYPE_PROBES

everything = np.ones(len(frame), dtype=bool)
svd = NgramSvd().fit(frame["query"])
x_all = np.concatenate([embeddings, svd.transform(frame["query"])], axis=1)
route = table.route_targets().to_numpy(dtype=np.float32)
cell = table.cell_targets.to_numpy(dtype=np.float32)
joined = pd.concat([table.corpus_targets(), table.outcome_rates(everything)], axis=1)
corpus = ((joined - joined.mean()) / joined.std().replace(0.0, 1.0).fillna(1.0)).to_numpy(np.float32)
TIE_POLICY = 1.0
tie_weights = np.where(frame["shape"] == "all_tied", TIE_POLICY, 1.0).astype(np.float32)

router = EncoderRouter().fit(x_all, route, cell, corpus, route_weights=tie_weights)
thresholds = tuned_thresholds(router.probabilities(x_all), frame)
print("tuned thresholds (sparse, dense) — rrf is the hedge when neither fires:",
      thresholds.round(2))

queries = [p.query for p in ARCHETYPE_PROBES]
encoder = SentenceTransformer(BGE)
probe_emb = encoder.encode(
    [EMBEDDING_PREFIXES[BGE] + q for q in queries], normalize_embeddings=True
)
probe_x = np.concatenate([probe_emb, svd.transform(pd.Series(queries))], axis=1)
served = router.predict_routes(probe_x.astype(np.float32), threshold=thresholds)

pd.DataFrame({
    "query": queries,
    "expected": [p.expected for p in ARCHETYPE_PROBES],
    "served": served,
    "agrees": [
        None if p.expected is None else str(p.expected) == s
        for p, s in zip(ARCHETYPE_PROBES, served)
    ],
})

In [ ]:
probs = router.probabilities(probe_x.astype(np.float32)).round(3)
probs.insert(0, "query", [q[:40] for q in queries])
probs

### 8b · Is the tuned `t_dense` a peak or a plateau?

Sweep `t_dense` while holding the tuned `t_sparse`, scoring each value with the
tuner's own cost-adjusted reward on the training differ rows. Reading the
`gap_to_best` column:

- **flat plateau** (several values within ~0.002 of the best): the tuner's pick
  is one point on a ridge — choosing the plateau's *low* edge is statistically
  free, and it lets mid-confidence dense beliefs (the 0.50-ish probes) serve.
- **steep peak**: the training data genuinely objects to a lower bar; the
  probes wait for v3 labels to re-price mid-confidence dense.

Honesty rail: never pick a threshold *because the probes look better* — that is
fitting the instrument. Only a flat plateau licenses the choice, and then only
as a tie-break within noise.

In [ ]:
from encoder_router.evaluate import COST_STEP
from encoder_router.table import HEAD_ROUTES, HEDGE, PRIORITY, serve_indices

differ = ((frame["shape"] == "routes_differ") & frame["serve"].notna()).to_numpy()
choices = [*HEAD_ROUTES, HEDGE]
rewards = (
    frame[[f"score_{r}" for r in choices]].to_numpy()[differ]
    - COST_STEP * np.array([PRIORITY.index(r) for r in choices])
)
ordered = router.probabilities(x_all)[list(HEAD_ROUTES)].to_numpy()[differ]

rows_ = []
for td in np.arange(0.30, 0.71, 0.05):
    trial = np.array([float(thresholds[0]), round(float(td), 2)])
    pick = serve_indices(ordered, trial)
    rows_.append({
        "t_dense": round(float(td), 2),
        "reward": float(rewards[np.arange(len(pick)), pick].mean()),
        "dense_share": float(
            (pick == list(HEAD_ROUTES).index("dense_only")).mean()
        ),
    })
curve = pd.DataFrame(rows_)
curve["gap_to_best"] = (curve["reward"].max() - curve["reward"]).round(4)
curve.round(4)

### 8c · Checkpoint the fitted router

A serving checkpoint is four artifacts: the trained net (`router.pt`), the
fitted n-gram SVD basis (`ngram_svd.joblib` — inputs are meaningless without
the exact basis), the tuned thresholds, and a `meta.json` recording when and
on what. Run this whenever a §8 fit is worth keeping — training is seeded and
reproducible, but a checkpoint survives kernel deaths and code changes alike.
The default path is overwritten on rerun; rename `CKPT` to keep several.

In [ ]:
import json
from datetime import datetime, timezone

CKPT = OUT_DIR / "checkpoints" / "probe_router"

router.save(CKPT / "router.pt")
svd.save(CKPT / "ngram_svd.joblib")
np.save(CKPT / "thresholds.npy", thresholds)
(CKPT / "meta.json").write_text(json.dumps({
    "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "embedding_model": BGE,
    "tie_policy": float(TIE_POLICY),
    "rows_trained": int(len(frame)),
    "thresholds": [float(t) for t in thresholds],
}, indent=2))
print("checkpoint written:")
for artifact in sorted(CKPT.iterdir()):
    print(f"  {artifact.name}")

# load-back, any later session (plus QueryEmbeddings(BGE) for the embedding):
#   from encoder_router.model import EncoderRouter
#   from encoder_router.table import NgramSvd
#   router = EncoderRouter.load(CKPT / "router.pt")
#   svd = NgramSvd.load(CKPT / "ngram_svd.joblib")
#   thresholds = np.load(CKPT / "thresholds.npy")

## 9 · Feature recoverability probe (diagnostic, zero gradient)

A ridge regression from the model's inputs to each taxonomy feature. Near-zero R²
means the feature is absent from the inputs — the only kind of feature that could
earn input status, and evidence for a rarity channel if the sparse-signals cluster
at the bottom. Nothing here touches the model.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

features = table.feature_matrix
x_tr, x_te, f_tr, f_te = train_test_split(x_all, features, test_size=0.25, random_state=0)
rows = []
for column in features.columns:
    if f_tr[column].std() == 0:
        continue
    score = Ridge(alpha=1.0).fit(x_tr, f_tr[column]).score(x_te, f_te[column])
    rows.append({"feature": column, "r2": score})
recoverability = pd.DataFrame(rows).sort_values("r2").reset_index(drop=True)
print("least recoverable (candidates for input status):")
display(recoverability.head(12))
print("most recoverable (provably present in the inputs):")
recoverability.tail(12)

## What to bring back

- design vs `no_branches` on `differ_agreement` — do the branches earn their place?
- design vs `shuffled_targets` — if they tie, the gain is regularization, not information.
- `lightgbm` vs design — if the trees match, ship the trees.
- `e5_control` vs design on holdout — the label-coupling verdict.

Not in this notebook: the fine-tuned-bge ceiling arm (own training loop, one run,
add here when wanted) and the cross-lane near-dup guard (open in TODOS). All of it
re-runs against v3 for the numbers of record.